**One Hot Encoding**<br>

This is the method used to encode the nominal categorical data in a dataset

**Steps**<br>

1. Make a seperate column for every category. These new columns we created are called as dummy variables
2. For every row we mark the category in its own columns as 1 and the remaining
3. eg. Let us say we have 3 categories: Yellow, Blue and Red. Then we will create 3 columns for each colour. Let us say for a row with category Blue the value in the columns will be either 0 or 1. 1 for the column with category that matches the category in that row while the others will be 0.

The problem with this approach is "multicollinearity" which occurs because of these dummy variables.

When the dummy vars have any mathematical relationship between them then they are dependent on each other which should not happen in ML as the input columns are also known as independent variables.<br>

Hence, we can say that multicollinearity occurs when 2 or more input features are highly correlated with each other.<br>

This makes it hard to determine the individual effect of each var and can make model coefficient unstable and unreliable.

Since the sum of the values in all the columns for a single row at a time is always '1' we can they are linearly dependent making it a perfect trap for multicollinearity.<br>

Solution: After making the n columns for n categories we drop the 1st new column we created. As a result we are left with (n-1) columns for the n categories.

OHE using most frequent columns:<br>

This method is used when there is high cardinality in categorical features, i.e the categorical feature has too many unique categories which would create too many dummy columns.<br>

In this approach we only keep the top N most frequent categories and we create dummy variables for these most frequent categories and group the non frequent columns in a new column (let us say "uncommon/other"). This helps reduce the dimensionality when we have too many categories.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('cars.csv')

In [3]:
df.head()

,brand,km_driven,fuel,owner,selling_price
0,Maruti,145500,Diesel,First Owner,450000
1,Skoda,120000,Diesel,Second Owner,370000
2,Honda,140000,Petrol,Third Owner,158000
3,Hyundai,127000,Diesel,First Owner,225000
4,Maruti,120000,Petrol,First Owner,130000


In [4]:
df.shape

(8128, 5)

In [5]:
df['fuel'].value_counts()

fuel
Diesel    4402
Petrol    3631
CNG         57
LPG         38
Name: count, dtype: int64

In [6]:
df['owner'].value_counts()

owner
First Owner             5289
Second Owner            2105
Third Owner              555
Fourth & Above Owner     174
Test Drive Car             5
Name: count, dtype: int64

**1. OneHotEncoding using Pandas**

In [7]:
n_cols=pd.get_dummies(df, columns=['fuel', 'owner'], sparse=True, dtype=int)   
#converts categories into dummy vars (dummy columns) with values 0 and 1

In [8]:
n_cols.sample(5)

,brand,km_driven,selling_price,fuel_CNG,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_First Owner,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
3637,Tata,185000,140000,0,1,0,0,0,0,1,0,0
5054,Maruti,60000,300000,0,0,0,1,1,0,0,0,0
3624,Hyundai,60000,525000,0,0,0,1,1,0,0,0,0
180,Volkswagen,13000,480000,0,0,0,1,1,0,0,0,0
2020,Honda,7800,850000,0,0,0,1,1,0,0,0,0


In [9]:
n_cols.shape

(8128, 12)

**2. K-1 OneHotEncoding**

In [10]:
n_minus_1_cols = pd.get_dummies(df, columns=['fuel', 'owner'], dtype=int, drop_first=True)

In [11]:
n_minus_1_cols.sample(5)

,brand,km_driven,selling_price,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
7119,Nissan,50000,325000,0,0,1,0,0,0,0
7777,Hyundai,47000,215000,0,0,1,0,1,0,0
1234,Tata,90000,55000,1,0,0,0,0,0,1
1804,Chevrolet,76000,170000,1,0,0,0,1,0,0
1820,Mahindra,80000,400000,1,0,0,0,0,0,0


In [12]:
n_minus_1_cols.shape

(8128, 10)

In [13]:
df.head()

,brand,km_driven,fuel,owner,selling_price
0,Maruti,145500,Diesel,First Owner,450000
1,Skoda,120000,Diesel,Second Owner,370000
2,Honda,140000,Petrol,Third Owner,158000
3,Hyundai,127000,Diesel,First Owner,225000
4,Maruti,120000,Petrol,First Owner,130000


**3. OneHotEncoding using Sklearn**

In [14]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,0:4], df.iloc[:,-1], test_size=0.2, random_state=2)

In [15]:
X_train.head()

,brand,km_driven,fuel,owner
5571,Hyundai,35000,Diesel,First Owner
2038,Jeep,60000,Diesel,First Owner
2957,Hyundai,25000,Petrol,First Owner
7618,Mahindra,130000,Diesel,Second Owner
6684,Hyundai,155000,Diesel,First Owner


In [16]:
from sklearn.preprocessing import OneHotEncoder

In [17]:
ohe = OneHotEncoder(drop='first', sparse_output=False,dtype=np.int32)

In [18]:
X_train_new = ohe.fit_transform(X_train[['fuel', 'owner']])

In [19]:
X_test_new = ohe.fit_transform(X_test[['fuel', 'owner']])

In [20]:
X_train_new.shape

(6502, 7)

In [21]:
X_train_new

array([[1, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       ...,
       [0, 0, 1, ..., 0, 0, 0],
       [1, 0, 0, ..., 1, 0, 0],
       [1, 0, 0, ..., 0, 0, 0]], shape=(6502, 7), dtype=int32)

In [22]:
np.hstack((X_train[['brand', 'km_driven']].values, X_train_new))

array([['Hyundai', 35000, 1, ..., 0, 0, 0],
       ['Jeep', 60000, 1, ..., 0, 0, 0],
       ['Hyundai', 25000, 0, ..., 0, 0, 0],
       ...,
       ['Tata', 15000, 0, ..., 0, 0, 0],
       ['Maruti', 32500, 1, ..., 1, 0, 0],
       ['Isuzu', 121000, 1, ..., 0, 0, 0]], shape=(6502, 9), dtype=object)

**4. OHE with Top Categories**

In [23]:
counts = df['brand'].value_counts()

In [24]:
df['brand'].nunique()
threshold = 100

In [25]:
repl = counts[counts <= threshold].index

In [26]:
pd.get_dummies(df['brand'].replace(repl, 'uncommon'), dtype=int).head()

,BMW,Chevrolet,Ford,Honda,Hyundai,Mahindra,Maruti,Renault,Skoda,Tata,Toyota,Volkswagen,uncommon
0,0,0,0,0,0,0,1,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,1,0,0,0,0
2,0,0,0,1,0,0,0,0,0,0,0,0,0
3,0,0,0,0,1,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,1,0,0,0,0,0,0
